# RQ1 Investigation: Why is OD=0 for 98% of samples?

**Purpose:** Manual investigation of the RQ1 Calibration results

**Key Finding to Investigate:**
- 98 out of 100 annotated samples have OD=0
- Spearman r = -0.126 (poor calibration)
- ECE = 0.690 (very high calibration error)

**What we'll examine side-by-side:**
1. Task description (what the agent was asked to do)
2. Original vs Perturbed step content (what was changed)
3. Human annotation (TSD, SER, TCS, notes)
4. Judge evaluation (overall_score, reasoning, step_errors)
5. OD computation (baseline_outcome, perturbed_outcome, reasoning)

In [1]:
import os
import json
from dotenv import load_dotenv
from pymongo import MongoClient
from IPython.display import display, HTML, Markdown
import pandas as pd

load_dotenv()

# Connect to MongoDB
uri = os.getenv('MONGODB_URI')
client = MongoClient(uri)
db = client['agent_judge_experiment']

print('Connected to MongoDB')

Connected to MongoDB


In [2]:
# Load RQ1 annotated perturbations (100 samples)
perturbations = list(db.perturbations.find({
    'selected_for_annotation': True
}))
print(f'Loaded {len(perturbations)} annotated perturbations')

# Load human annotations
pert_ids = [p['perturbation_id'] for p in perturbations]
annotations = list(db.annotations.find({
    'perturbation_id': {'$in': pert_ids}
}))
ann_by_pert = {a['perturbation_id']: a for a in annotations}
print(f'Loaded {len(annotations)} human annotations')

# Load judge evaluations
# Judge evaluations are keyed by perturbed_trajectory_id
perturbed_traj_ids = [p['perturbed_trajectory_id'] for p in perturbations]
judge_evals = list(db.judge_evaluations.find({
    'trajectory_id': {'$in': perturbed_traj_ids}
}))
judge_by_traj = {e['trajectory_id']: e for e in judge_evals}
print(f'Loaded {len(judge_evals)} judge evaluations')

Loaded 100 annotated perturbations
Loaded 100 human annotations
Loaded 100 judge evaluations


In [3]:
# Quick stats on OD values
od_values = [p.get('od', {}).get('value', None) for p in perturbations]
od_nonzero = [v for v in od_values if v is not None and v != 0]

print('OD Distribution:')
print(f'  Total samples: {len(perturbations)}')
print(f'  With OD computed: {len([v for v in od_values if v is not None])}')
print(f'  OD = 0: {len([v for v in od_values if v == 0])}')
print(f'  OD != 0: {len(od_nonzero)}')
if od_nonzero:
    print(f'  Non-zero OD values: {od_nonzero}')

OD Distribution:
  Total samples: 100
  With OD computed: 100
  OD = 0: 98
  OD != 0: 2
  Non-zero OD values: [0.1, -0.3]


---
## Helper Function: Display a Sample

In [ ]:
def display_sample(idx):
    """Display all information about a sample for manual review."""
    p = perturbations[idx]
    pert_id = p['perturbation_id']
    
    # Get related data
    ann = ann_by_pert.get(pert_id, {})
    judge = judge_by_traj.get(p['perturbed_trajectory_id'], {})
    od = p.get('od', {})
    
    # Load original trajectory for task description
    orig_traj = db.trajectories.find_one({'trajectory_id': p['original_trajectory_id']})
    task_desc = orig_traj.get('ground_truth', {}).get('task_description', 'N/A') if orig_traj else 'N/A'
    expected_answer = orig_traj.get('ground_truth', {}).get('expected_answer', 'N/A') if orig_traj else 'N/A'
    
    # Build HTML display
    html = f"""
    <div style="border: 2px solid #333; padding: 15px; margin: 10px 0; border-radius: 8px;">
    
    <h2>Sample {idx + 1}: {pert_id[:50]}...</h2>
    
    <table style="width: 100%; border-collapse: collapse;">
    <tr>
        <td style="width: 150px; font-weight: bold;">Benchmark:</td>
        <td>{p.get('benchmark', orig_traj.get('benchmark', 'unknown') if orig_traj else 'unknown')}</td>
    </tr>
    <tr>
        <td style="font-weight: bold;">Perturbation Type:</td>
        <td style="color: {'red' if p.get('perturbation_type') == 'planning' else 'blue'};">{p.get('perturbation_type', 'unknown')}</td>
    </tr>
    <tr>
        <td style="font-weight: bold;">Position:</td>
        <td>{p.get('perturbation_position', 'unknown')}</td>
    </tr>
    <tr>
        <td style="font-weight: bold;">Step Number:</td>
        <td>{p.get('perturbed_step_number', 'unknown')}</td>
    </tr>
    </table>
    
    <hr>
    <h3>Task Description</h3>
    <div style="background: #f5f5f5; padding: 10px; border-radius: 5px; max-height: 200px; overflow-y: auto;">
        <pre style="white-space: pre-wrap;"><font color="black">{task_desc[:1000]}{'...' if len(task_desc) > 1000 else ''}</font></pre>
    </div>
    
    <h3>Expected Answer (Ground Truth)</h3>
    <div style="background: #e8f5e9; padding: 10px; border-radius: 5px; max-height: 150px; overflow-y: auto;">
        <pre style="white-space: pre-wrap;"><font color="black">{str(expected_answer)[:500]}{'...' if len(str(expected_answer)) > 500 else ''}</font></pre>
    </div>
    
    <hr>
    <h3>Original vs Perturbed Step Content</h3>
    <table style="width: 100%;">
    <tr>
        <th style="width: 50%; background: #c8e6c9; padding: 5px;">ORIGINAL</th>
        <th style="width: 50%; background: #ffcdd2; padding: 5px;">PERTURBED</th>
    </tr>
    <tr>
        <td style="vertical-align: top; padding: 10px; background: #f1f8e9;">
            <pre style="white-space: pre-wrap; font-size: 11px; max-height: 300px; overflow-y: auto;"><font color="black">{p.get('original_step_content', 'N/A')[:1500]}</font></pre>
        </td>
        <td style="vertical-align: top; padding: 10px; background: #ffebee;">
            <pre style="white-space: pre-wrap; font-size: 11px; max-height: 300px; overflow-y: auto;"><font color="black">{p.get('perturbed_step_content', 'N/A')[:1500]}</font></pre>
        </td>
    </tr>
    </table>
    
    <hr>
    <h3>Evaluation Results</h3>
    <table style="width: 100%; border-collapse: collapse;">
    <tr style="background: #e3f2fd;">
        <th style="padding: 10px; border: 1px solid #ccc;"><font color="black">Human Annotation</font></th>
        <th style="padding: 10px; border: 1px solid #ccc;"><font color="black">Judge Evaluation</font></th>
        <th style="padding: 10px; border: 1px solid #ccc;"><font color="black">OD Computation</font></th>
    </tr>
    <tr>
        <td style="vertical-align: top; padding: 10px; border: 1px solid #ccc;">
            <b>Task Failure (TSD):</b> {ann.get('task_success_degradation', 'N/A')}<br>
            <b>Errors After (SER):</b> {ann.get('subsequent_error_rate', 'N/A')}<br>
            <b>TCS Score:</b> {ann.get('tcs_score', 'N/A')}<br>
            <b>Notes:</b><br>
            <div style="background: #fff3e0; padding: 5px; font-size: 11px;">{ann.get('notes', 'N/A')}</div>
        </td>
        <td style="vertical-align: top; padding: 10px; border: 1px solid #ccc;">
            <b>Overall Score:</b> {judge.get('overall_score', 'N/A')}<br>
            <b>JPS (100-score):</b> {100 - judge.get('overall_score', 50) if judge.get('overall_score') else 'N/A'}<br>
            <b>Step Errors Found:</b> {len(judge.get('step_errors', []))}<br>
            <b>Reasoning:</b><br>
            <div style="background: #e8eaf6; padding: 5px; font-size: 11px; max-height: 150px; overflow-y: auto;"><font color="black">{judge.get('reasoning', 'N/A')[:500]}</font></div>
        </td>
        <td style="vertical-align: top; padding: 10px; border: 1px solid #ccc;">
            <b>Baseline Outcome:</b> {od.get('baseline_outcome', 'N/A')}<br>
            <b>Perturbed Outcome:</b> {od.get('perturbed_outcome', 'N/A')}<br>
            <b style="color: {'red' if od.get('value', 0) != 0 else 'gray'};">OD Value:</b> {od.get('value', 'N/A')}<br>
            <hr>
            <b>Baseline Reasoning:</b><br>
            <div style="background: #e8f5e9; padding: 5px; font-size: 10px; max-height: 100px; overflow-y: auto;"><font color="black">{od.get('baseline_reasoning', 'N/A')}</font></div>
            <b>Perturbed Reasoning:</b><br>
            <div style="background: #ffebee; padding: 5px; font-size: 10px; max-height: 100px; overflow-y: auto;"><font color="black">{od.get('perturbed_reasoning', 'N/A')}</font></div>
        </td>
    </tr>
    </table>
    
    </div>
    """
    display(HTML(html))

---
## View Samples with OD != 0 (the interesting cases)

In [17]:
# Find samples where OD is non-zero
nonzero_indices = [i for i, p in enumerate(perturbations) 
                   if p.get('od', {}).get('value', 0) != 0]

print(f'Found {len(nonzero_indices)} samples with OD != 0')
for i in nonzero_indices:
    p = perturbations[i]
    print(f'  [{i}] OD={p.get("od", {}).get("value", "N/A"):.3f}, '
          f'type={p.get("perturbation_type")}, pos={p.get("perturbation_position")}')

Found 2 samples with OD != 0
  [5] OD=0.100, type=parameter, pos=early
  [88] OD=-0.300, type=planning, pos=late


In [18]:
# Display non-zero OD samples
for i in nonzero_indices[:5]:  # Show first 5
    display_sample(i)

---
## View Samples where Human said TSD=1 but OD=0

In [ ]:
# Find samples where human said task failed (TSD=1) but OD=0
# This is the most suspicious case - human says it failed but OD grader disagrees

disagreement_indices = []
for i, p in enumerate(perturbations):
    pert_id = p['perturbation_id']
    ann = ann_by_pert.get(pert_id, {})
    od_val = p.get('od', {}).get('value', 0)
    tsd = ann.get('task_success_degradation', 0)
    
    if tsd == 1 and od_val == 0:
        disagreement_indices.append(i)

print(f'Found {len(disagreement_indices)} samples where Human TSD=1 but OD=0')
print('\nThese are cases where human annotator said the perturbation caused task failure,')
print('but the OD grader gave both baseline and perturbed the same score.')

In [ ]:
# Display disagreement samples
for i in disagreement_indices[:5]:  # Show first 5
    display_sample(i)

---
## View random OD=0 samples to understand the pattern

In [ ]:
# View a few random OD=0 samples
import random
random.seed(42)

zero_indices = [i for i, p in enumerate(perturbations) 
                if p.get('od', {}).get('value', 0) == 0]

sample_zero = random.sample(zero_indices, min(5, len(zero_indices)))
print(f'Randomly sampling {len(sample_zero)} samples from {len(zero_indices)} with OD=0')

In [ ]:
for i in sample_zero:
    display_sample(i)

---
## Deep Dive: OD Grading Distribution

In [ ]:
# Analyze what scores the OD grader is giving
baseline_scores = []
perturbed_scores = []

for p in perturbations:
    od = p.get('od', {})
    if od.get('baseline_outcome') is not None:
        baseline_scores.append(od['baseline_outcome'])
        perturbed_scores.append(od['perturbed_outcome'])

print('OD Grader Score Distribution:')
print(f'\nBaseline Outcomes:')
print(f'  Mean: {sum(baseline_scores)/len(baseline_scores):.1f}')
print(f'  Min:  {min(baseline_scores)}')
print(f'  Max:  {max(baseline_scores)}')
print(f'\nPerturbed Outcomes:')
print(f'  Mean: {sum(perturbed_scores)/len(perturbed_scores):.1f}')
print(f'  Min:  {min(perturbed_scores)}')
print(f'  Max:  {max(perturbed_scores)}')

# How many have identical scores?
identical = sum(1 for b, p in zip(baseline_scores, perturbed_scores) if b == p)
print(f'\nIdentical baseline/perturbed scores: {identical}/{len(baseline_scores)} ({identical/len(baseline_scores)*100:.1f}%)')

In [ ]:
# Histogram of scores
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(baseline_scores, bins=10, range=(0, 100), alpha=0.7, label='Baseline', color='green')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Baseline Outcome Scores')

axes[1].hist(perturbed_scores, bins=10, range=(0, 100), alpha=0.7, label='Perturbed', color='red')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Perturbed Outcome Scores')

# Scatter: baseline vs perturbed
axes[2].scatter(baseline_scores, perturbed_scores, alpha=0.5)
axes[2].plot([0, 100], [0, 100], 'r--', label='y=x (OD=0 line)')
axes[2].set_xlabel('Baseline Score')
axes[2].set_ylabel('Perturbed Score')
axes[2].set_title('Baseline vs Perturbed')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## Analysis: Why is OD=0 so common?

Based on the data above, analyze the possible causes:

1. **Both get same high score (e.g., 80-100)**: The grader is too lenient and gives high scores regardless of quality
2. **Both get same low score**: The baseline trajectories may already be failing, so perturbation doesn't make it worse
3. **Final answer extraction issue**: OD grader may not be extracting the right "final answer"
4. **Perturbations don't affect final output**: The errors introduced may not propagate to change the final answer

In [ ]:
# Group by score buckets to understand the pattern
from collections import Counter

# Count (baseline_bucket, perturbed_bucket) pairs
bucket_pairs = Counter()
for b, p in zip(baseline_scores, perturbed_scores):
    b_bucket = int(b // 20) * 20  # 0-20, 20-40, etc.
    p_bucket = int(p // 20) * 20
    bucket_pairs[(b_bucket, p_bucket)] += 1

print('Score Bucket Pairs (baseline, perturbed) -> count:')
for pair, count in sorted(bucket_pairs.items(), key=lambda x: -x[1]):
    print(f'  {pair}: {count}')

---
## View specific sample by index

In [ ]:
# Change this index to view any specific sample
SAMPLE_INDEX = 0
display_sample(SAMPLE_INDEX)

---
## Summary Table: All 100 Samples

In [ ]:
# Create summary DataFrame
summary_data = []

for i, p in enumerate(perturbations):
    pert_id = p['perturbation_id']
    ann = ann_by_pert.get(pert_id, {})
    judge = judge_by_traj.get(p['perturbed_trajectory_id'], {})
    od = p.get('od', {})
    
    summary_data.append({
        'idx': i,
        'type': p.get('perturbation_type', 'unknown'),
        'position': p.get('perturbation_position', 'unknown'),
        'TSD': ann.get('task_success_degradation', 'N/A'),
        'SER': ann.get('subsequent_error_rate', 'N/A'),
        'TCS': ann.get('tcs_score', 'N/A'),
        'JPS': 100 - judge.get('overall_score', 50) if judge.get('overall_score') else 'N/A',
        'Baseline': od.get('baseline_outcome', 'N/A'),
        'Perturbed': od.get('perturbed_outcome', 'N/A'),
        'OD': od.get('value', 'N/A'),
    })

df_summary = pd.DataFrame(summary_data)
df_summary

In [ ]:
# Filter to show only samples where Human TSD=1
df_summary[df_summary['TSD'] == 1]

In [ ]:
# Close connection
client.close()
print('Done!')